In [27]:
import sys
!{sys.executable} -m pip install mediapipe==1.1.1


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [33]:
import sys
!{sys.executable} -m pip install scikit-learn pandas

   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------- ----------------------------- 2.1/8.4 MB 10.8 MB/s eta 0:00:01
   -------------------- ------------------- 4.2/8.4 MB 10.9 MB/s eta 0:00:01
   ------------------------------- -------- 6.6/8.4 MB 10.8 MB/s eta 0:00:01
   -------------------------------------- - 8.1/8.4 MB 10.8 MB/s eta 0:00:01
   ---------------------------------------- 8.4/8.4 MB 9.2 MB/s  0:00:00
   ---------------------------------------- 0.0/37.4 MB ? eta -:--:--
   -- ------------------------------------- 2.1/37.4 MB 10.8 MB/s eta 0:00:04
   ---- ----------------------------------- 4.5/37.4 MB 10.9 MB/s eta 0:00:04
   ------- -------------------------------- 6.6/37.4 MB 10.9 MB/s eta 0:00:03
   --------- ------------------------------ 8.7/37.4 MB 10.9 MB/s eta 0:00:03
   ----------- ---------------------------- 11.0/37.4 MB 10.8 MB/s eta 0:00:03
   -------------- ------------------------- 13.1/37.4 MB 10.8 MB/s eta 0:00:03
   -------


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [49]:
import cv2
import mediapipe as mp
import os
import pandas as pd
import urllib.request
import csv

# 1. Unduh file model task MediaPipe jika belum ada
model_path = 'hand_landmarker.task'
if not os.path.exists(model_path):
    url = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
    urllib.request.urlretrieve(url, model_path)

BaseOptions = mp.tasks.BaseOptions
VisionRunningMode = mp.tasks.vision.RunningMode
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions

options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.IMAGE,
    num_hands=1
)

# Tentukan path folder dataset gambar Anda di sini (sesuaikan dengan nama folder di komputer Anda)
dataset_dir = 'asl_alphabet_train' # Ganti jika nama folder Anda berbeda
dataset = []
MAX_DATA_PER_FOLDER = 150 # Batasan maksimal data per huruf

print("[INFO] Memulai ekstraksi titik koordinat (maksimal 60 data per huruf)...")

with HandLandmarker.create_from_options(options) as landmarker:
    # Melakukan looping berdasarkan nama folder (A-Z)
    for label_dir in sorted(os.listdir(dataset_dir)):
        label_path = os.path.join(dataset_dir, label_dir)
        
        if os.path.isdir(label_path):
            count = 0  # Penghitung data per folder
            print(f"Memproses folder huruf: {label_dir.upper()}")
            
            for img_name in os.listdir(label_path):
                # Jika sudah mencapai 60 data untuk huruf ini, hentikan dan lanjut ke huruf berikutnya
                if count >= MAX_DATA_PER_FOLDER:
                    break
                
                img_path = os.path.join(label_path, img_name)
                
                # Baca gambar menggunakan OpenCV
                img = cv2.imread(img_path)
                if img is None:
                    continue
                
                # Konversi ke format MediaPipe
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
                detection_result = landmarker.detect(mp_image)
                
                # Jika tangan terdeteksi dalam gambar, ambil 21 titik koordinatnya
                if detection_result.hand_landmarks:
                    for hand_landmarks in detection_result.hand_landmarks:
                        row = []
                        for lm in hand_landmarks:
                            row.extend([lm.x, lm.y])
                        
                        # Tambahkan label hurufnya di akhir baris
                        row.append(label_dir.upper())
                        dataset.append(row)
                        count += 1  # Tambah hitungan data yang berhasil diambil

# 2. Simpan hasil ekstraksi ke file CSV baru
if len(dataset) > 0:
    output_filename = 'asl_dataset_balanced_60.csv'
    with open(output_filename, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([f"x{i}" for i in range(21)] + [f"y{i}" for i in range(21)] + ['label'])
        writer.writerows(dataset)
    print(f"\nBerhasil! Total {len(dataset)} data terkumpul dan disimpan ke '{output_filename}' (Maksimal 60 per huruf).")
else:
    print("Tidak ada landmark tangan yang terdeteksi pada gambar-gambar tersebut.")

[INFO] Memulai ekstraksi titik koordinat (maksimal 60 data per huruf)...
Memproses folder huruf: A
Memproses folder huruf: B
Memproses folder huruf: C
Memproses folder huruf: D
Memproses folder huruf: E
Memproses folder huruf: F
Memproses folder huruf: G
Memproses folder huruf: H
Memproses folder huruf: I
Memproses folder huruf: J
Memproses folder huruf: K
Memproses folder huruf: L
Memproses folder huruf: M
Memproses folder huruf: N
Memproses folder huruf: O
Memproses folder huruf: P
Memproses folder huruf: Q
Memproses folder huruf: R
Memproses folder huruf: S
Memproses folder huruf: T
Memproses folder huruf: U
Memproses folder huruf: V
Memproses folder huruf: W
Memproses folder huruf: X
Memproses folder huruf: Y
Memproses folder huruf: Z

Berhasil! Total 3900 data terkumpul dan disimpan ke 'asl_dataset_balanced_60.csv' (Maksimal 60 per huruf).


In [50]:
import cv2
import mediapipe as mp
import pandas as pd
import numpy as np
import joblib
import os
import urllib.request
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# ==========================================
# TAHAP 1: TRAINING MODEL DENGAN DATASET BARU
# ==========================================
csv_filename = 'asl_dataset_balanced_60.csv'
model_filename = 'asl_random_forest_balanced.pkl'

if not os.path.exists(csv_filename):
    print(f"Error: File '{csv_filename}' tidak ditemukan! Pastikan Anda sudah menjalankan skrip ekstraksi sebelumnya.")
else:
    print(f"[INFO] Memuat data dari '{csv_filename}'...")
    df = pd.read_csv(csv_filename)
    
    # Pisahkan fitur (koordinat x, y) dan label (huruf)
    X = df.drop('label', axis=1)
    y = df['label']
    
    print(f"Total baris data: {len(df)}")
    print(f"Daftar huruf yang dilatih: {sorted(y.unique())}")
    
    # Bagi data untuk training (80%) dan testing (20%)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Latih model Random Forest
    print("[INFO] Sedang melatih model Random Forest...")
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    # Evaluasi akurasi
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"[SUKSES] Model berhasil dilatih! Akurasi pengujian: {acc * 100:.2f}%")
    
    # Simpan model
    joblib.dump(model, model_filename)
    print(f"[INFO] Model disimpan sebagai '{model_filename}'\n")

# ==========================================
# TAHAP 2: REAL-TIME DETECTION DENGAN KAMERA
# ==========================================
# Unduh file model task MediaPipe jika belum ada
model_path = 'hand_landmarker.task'
if not os.path.exists(model_path):
    print("Mengunduh model hand_landmarker.task...")
    url = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
    urllib.request.urlretrieve(url, model_path)

# Inisialisasi MediaPipe Hand Landmarker
BaseOptions = mp.tasks.BaseOptions
VisionRunningMode = mp.tasks.vision.RunningMode
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions

options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.IMAGE,
    num_hands=1
)

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Kamera tidak dapat diakses.")
else:
    print("Mulai Deteksi Real-Time! Arahkan tangan ke kamera. Tekan 'q' untuk keluar.")
    
    with HandLandmarker.create_from_options(options) as landmarker:
        try:
            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    break
                
                # Flip horizontal agar seperti cermin
                frame = cv2.flip(frame, 1)
                
                # Konversi ke format MediaPipe Image
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                detection_result = landmarker.detect(mp_image)
                
                display_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                predicted_label = "Menunggu Tangan..."
                
                if detection_result.hand_landmarks:
                    for hand_landmarks in detection_result.hand_landmarks:
                        row = []
                        for lm in hand_landmarks:
                            row.extend([lm.x, lm.y])
                        
                        # Prediksi menggunakan model yang baru dilatih
                        input_data = np.array([row])
                        prediction = model.predict(input_data)
                        predicted_label = f"Huruf: {prediction[0]}"
                
                # Tampilkan hasil prediksi di layar
                cv2.putText(display_frame, predicted_label, (30, 50), 
                            cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3, cv2.LINE_AA)
                
                cv2.imshow("Uji Coba Model ASL", cv2.cvtColor(display_frame, cv2.COLOR_RGB2BGR))
                
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    print("[INFO] Pengujian dihentikan.")
                    break
                    
        except KeyboardInterrupt:
            print("[INFO] Dihentikan paksa.")
            
        finally:
            cap.release()
            cv2.destroyAllWindows()

[INFO] Memuat data dari 'asl_dataset_balanced_60.csv'...
Total baris data: 3900
Daftar huruf yang dilatih: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']
[INFO] Sedang melatih model Random Forest...
[SUKSES] Model berhasil dilatih! Akurasi pengujian: 98.33%
[INFO] Model disimpan sebagai 'asl_random_forest_balanced.pkl'

Mulai Deteksi Real-Time! Arahkan tangan ke kamera. Tekan 'q' untuk keluar.


c:\Users\nathanael\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\nathanael\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\nathanael\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\nathanael\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\nathanael\AppData\Local\Python\pythoncore-3.14-64\L

[INFO] Dihentikan paksa.
